# Stage 12 · Diagnose the white Chair render

Attach the original **Chair kit** and **chair-s42-stage11-review.zip** (zipped or expanded). Choose **GPU T4 ×2** and **Run All**. Two isolated diagnostic copies start from the saved regularised update-zero weights, optimizer and CPU/CUDA random states. Both keep frequency annealing. One uses the original near-camera weight 0.01 and the other uses 0. Both use training cameras only.

Each copy saves the actual weights, preview and gradient measurements at **128** and **500**. In these bounded copies the fixed-ray stop is logged as an observation, allowing us to see whether it persists. Nonfinite loss or diagnostics still stop the job. The original Stage 11 run and the two step-500 vanilla checkpoints stay in the input archive.

From the short Stage 11 timing, expect roughly **1½ minutes of optimiser work** in parallel. Allow **4–8 minutes total** with setup, probes and exports. Save the 128-update backup. Send **chair-s42-stage12-review.zip** even when one or both predictions stay white. If a process fails, send the diagnostics ZIP and traceback. In a fresh session, attach the latest completed Stage 12 snapshot alongside the two original inputs.

In [ ]:
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
import torch
from IPython.display import HTML, display

inputs, outputs = Path("/kaggle/input"), Path("/kaggle/working")
work = outputs/"chair-stage12"
work.mkdir(exist_ok=True, parents=True)
if "stage12_temp" not in globals():
    stage12_temp = TemporaryDirectory(prefix="chair-probe-")
temp = Path(stage12_temp.name)
lanes = {"full": 0.01, "frequency-only": 0.0}
stops = (128, 500)
start_sha = "f88c4b4fcec9ffec8b34004eaa14d2e5939c266c28df6e2dca304cec9cfe582c"
review_sha = "80b250038e7335b82e982e4f576538f9128ef449a65650a3435f697a81040db5"
data_hash = "9c0224042d8d8060a2684ad70e6edfdf74f7b5acbd159aaa90142581c4e561e4"
if torch.cuda.device_count() < 2:
    raise RuntimeError("Select GPU T4 ×2 and restart this notebook.")
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = [n.strip() for n in visible.split(",")][:2] if visible else ["0", "1"]
assert len(set(devices)) == 2
print("Diagnostic workers:", [torch.cuda.get_device_name(i) for i in range(2)])

def read(path):
    return json.loads(path.read_bytes())

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def unpack(path):
    dest = temp/hashlib.sha256(str(path).encode()).hexdigest()
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        assert len(z.namelist()) == len(set(z.namelist()))
        for name in z.namelist():
            assert not Path(name).is_absolute() and ".." not in Path(name).parts
            z.extract(name, dest)
    return dest

def source(marker, filename, valid):
    for p in sorted(inputs.rglob(marker)):
        if valid(p.parent):
            return p.parent
    for p in sorted(inputs.rglob(filename)):
        d = unpack(p)
        if valid(d):
            return d
    raise FileNotFoundError(f"Attach the reviewed {filename} or its expanded contents.")

repo = source("RECOVERY_CHAIR.txt", "chair-input.zip", lambda d: read(d/"RECOVERY_CHAIR.txt")["data_hash"] == data_hash)
original = source("stage11-review.json", "chair-s42-stage11-review.zip", lambda d: sha(d/"stage11-review.json") == review_sha)
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.health import dead_field
from recovery.media import digest, write_json
from recovery.training import fingerprint, implementation_hash, load_state, settings

review = read(original/"stage11-review.json")
assert implementation_hash() == review["implementation_hash"] and review["data_hash"] == data_hash
for name, expected in review["sha256"].items():
    assert (original/name).resolve().is_relative_to(original.resolve()) and sha(original/name) == expected
parent = original/"runs/chair/sparse-regularized/seed-42"
parent_meta = read(parent/"run.json")
initial, failure = read(parent/"initial-health.json"), read(parent/"health.json")
assert sha(parent/"checkpoint.pt") == start_sha and load_state(parent,parent_meta)["step"] == 0
assert failure["step"] == 128 and dead_field(failure) and initial["step"] == 0
manifest = work/"chair.json"
scene = verify_manifest(original/"manifests/chair.json", repo/"data/blender/chair")
assert scene["data_hash"] == data_hash and scene["splits"]["sparse"] == [26,86,2,55,75,93,16,73]
write_json(manifest,scene)
cfg = settings("main", scene["camera"])
assert cfg == parent_meta["training_config"] == review["matched_config"]["prefix_config"]
print("Verified two vanilla checkpoints at 500 and the regularised start at 0. Recorded failure at 128.")

In [ ]:
worker_code = """
import hashlib
import json
import sys
from pathlib import Path
from unittest.mock import patch
import torch
from recovery import training
from recovery.health import dead_field, diagnostics, fixed_rays
from recovery.media import write_json
from sparse_nerf.data import SceneDataset

def probe(model, dataset, rays, progress, coefficient):
    o,d,target,fg = rays
    captured = {}
    def capture(name):
        def save(module, args, values):
            captured[name] = values.detach()
        return save
    handles = [field.color.register_forward_hook(capture(name)) for name,field in (("coarse",model.coarse),("fine",model.fine))]
    try:
        out = model(o,d,dataset.near,dataset.far,regularized=True,progress=progress,
                    randomized=False,white_background=dataset.white_background)
    finally:
        for h in handles:
            h.remove()
    stats = {}
    for name,field,suffix in (("coarse",model.coarse,"_coarse"),("fine",model.fine,"")):
        rgb = out["rgb"+suffix][fg]
        mse = (rgb-target[fg]).square().mean()
        density,color = torch.autograd.grad(mse,(field.density.weight,field.color.weight),retain_graph=True)
        logits = captured[name][fg]
        assert all(t.isfinite().all() for t in (density,color,logits,rgb))
        stats[name] = dict(fg_rgb_min=rgb.min().item(), fg_rgb_mean=rgb.mean(0).tolist(),
            white_rgb_channel_fraction=(rgb==1).float().mean().item(),
            sample_sigmoid_exactly_one_fraction=(logits.sigmoid()==1).float().mean().item(),
            sample_logit_min=logits.min().item(), sample_logit_max=logits.max().item(),
            density_head_rgb_grad_l2_fp64=density.double().norm().item(),
            colour_head_rgb_grad_l2_fp64=color.double().norm().item(),
            colour_head_rgb_grad_nonzero=int(color.count_nonzero().item()))
        if name=="fine":
            near_grad, = torch.autograd.grad(out["near_loss"],field.density.weight)
            assert near_grad.isfinite().all()
            stats[name].update(near_loss=out["near_loss"].item(),weighted_near_loss=coefficient*out["near_loss"].item(),
                               near_density_head_grad_l2_unweighted=near_grad.double().norm().item(),
                               near_density_head_grad_l2_weighted=coefficient*near_grad.double().norm().item())
    return stats

def run(run_dir, manifest, stop, device):
    assert 0 < stop <= 500
    meta = json.loads((run_dir/"run.json").read_text(encoding="utf-8"))
    cfg = meta["training_config"]
    assert meta["stage12_diagnostic"] and meta["protocol"]=="sparse-regularized"
    checks = []
    def observe(info):
        checks.append(dict(step=info["step"], dead_on_fixed_rays=dead_field(info)))
        return False
    # This isolated process observes the original stop condition at the same
    # updates. The bounded copy still fails on nonfinite loss or diagnostics.
    with patch.object(training,"dead_field",observe):
        training.train(run_dir,manifest,meta["protocol"],meta["seed"],device,cfg,
                       resume=True,stop_at=stop,save_every=500)
    state=training.load_state(run_dir,meta)
    assert state["step"]==stop
    model=training.make_model(cfg,device)
    model.load_state_dict(state["model"])
    dataset=SceneDataset(manifest,"sparse",cfg["downscale"])
    progress=(stop-1)/(cfg["steps"]-1)
    evidence={}
    for label,count in (("original",16),("expanded",32)):
        rays=fixed_rays(dataset,list(range(len(dataset))),device,pixels_per_class=count)
        health=diagnostics(model,dataset,rays,progress,True)
        evidence[label]=dict(health=health, dead_on_fixed_rays=dead_field(health),
                             measured=probe(model,dataset,rays,progress,cfg["occ_weight"]))
    baseline=json.loads((run_dir/"initial-health.json").read_text(encoding="utf-8"))
    h=evidence["original"]["health"];fine=h["fine"]
    passed=(not dead_field(h) and fine["region_psnr"]>h["white_region_psnr"]+3 and
            fine["region_mse"]<baseline["fine"]["region_mse"] and fine["region_opacity"]>.02)
    rng={k: hashlib.sha256(state[k].numpy().tobytes()).hexdigest()
         for k in ("cpu_rng","cuda_rng") if state[k] is not None}
    result=dict(step=stop,lane=meta["stage12_diagnostic"],checks=checks,probes=evidence,
                passed_foreground_check=passed,rng_sha256=rng)
    write_json(run_dir/"probe.json",result)
    print("Probe saved",stop,"fixed foreground PSNR",fine["region_psnr"],"opacity",fine["region_opacity"],
          "colour gradient",fine["color_grad"],"passed",passed,flush=True)

if __name__=="__main__":
    run(Path(sys.argv[1]),Path(sys.argv[2]),int(sys.argv[3]),torch.device("cuda:0"))
"""
worker_path = temp/"chair_probe.py"
worker_path.write_text(worker_code, encoding="utf-8")

In [ ]:
experiment = digest(dict(parent=start_sha, lanes=lanes, stops=stops, worker=worker_code))
def child_meta(lane):
    return dict(parent_meta, run_id=digest([experiment,lane])[:32],stage12_diagnostic=lane,
                parent_run_id=parent_meta["run_id"],parent_step0_sha256=start_sha,
                training_config=dict(cfg,occ_weight=lanes[lane]))

def relative(lane,step):
    return Path("diagnostics")/lane/f"step-{step:06d}"

def members(step):
    return ("run.json","checkpoint.pt","training.jsonl","initial-health.json","health.json",
            "probe.json","console.log",f"train-{step:06d}.png")

def checked(folder,lane,step):
    assert read(folder/"run.json")==child_meta(lane)
    s=load_state(folder,child_meta(lane))
    assert s["step"]==step and all(s[k].numel() for k in ("cpu_rng","cuda_rng"))
    slots=s["optimizer"]["state"]
    assert len(slots)==48 and all(v["step"].item()==step for v in slots.values())
    assert all(t.isfinite().all() for t in [*s["model"].values(),*(v[k] for v in slots.values() for k in ("exp_avg","exp_avg_sq"))])
    p=read(folder/"probe.json")
    assert (p["step"],p["lane"])==(step,lane) and read(folder/"health.json")["step"]==step
    assert p["rng_sha256"]=={k: hashlib.sha256(s[k].numpy().tobytes()).hexdigest() for k in ("cpu_rng","cuda_rng")}
    assert all((folder/name).is_file() for name in members(step))
    return p

def restore_common(folder):
    r=read(folder/"stage12-review.json")
    if r.get("tag")=="diagnostics":
        return
    if r["experiment"]!=experiment or r["source_review_sha256"]!=review_sha:
        raise ValueError("Attached probe snapshot belongs to another experiment.")
    for name,expected in r["sha256"].items():
        p=(folder/name).resolve()
        assert p.is_relative_to(folder.resolve()) and sha(p)==expected
    for lane in lanes:
        for step in stops:
            rel=relative(lane,step); src=folder/rel
            if not (src/"probe.json").is_file():
                continue
            assert all((rel/n).as_posix() in r["sha256"] for n in members(step))
            checked(src,lane,step)
            dest=work/rel
            if dest.exists():
                checked(dest,lane,step)
                assert sha(dest/"checkpoint.pt")==sha(src/"checkpoint.pt")
                continue
            dest.parent.mkdir(exist_ok=True,parents=True)
            with TemporaryDirectory(dir=dest.parent) as staging:
                run=Path(staging)/"run";run.mkdir()
                for name in members(step):
                    shutil.copyfile(src/name,run/name)
                run.rename(dest)

for path in sorted(inputs.rglob("stage12-review.json")):
    restore_common(path.parent)
for path in sorted(inputs.rglob("chair-s42-stage12-*.zip")):
    if not path.name.endswith("diagnostics.zip"):
        restore_common(unpack(path))

def seed_copy(lane,step,run):
    meta=child_meta(lane)
    if step==stops[0]:
        s=load_state(parent,parent_meta)
        # Only the fork's metadata fingerprint changes; both RNG states and every learned tensor are copied.
        s["fingerprint"]=fingerprint(meta)
        torch.save(s,run/"checkpoint.pt")
        shutil.copyfile(parent/"initial-health.json",run/"initial-health.json")
    else:
        src=work/relative(lane,stops[0])
        checked(src,lane,stops[0])
        for name in ("checkpoint.pt","initial-health.json","training.jsonl"):
            shutil.copyfile(src/name,run/name)
    write_json(run/"run.json",meta)

def snapshot(tag, extra=()):
    files={}
    for lane in lanes:
        for step in stops:
            rel=relative(lane,step)
            if (work/rel).is_dir():
                if tag!="diagnostics":
                    checked(work/rel,lane,step)
                for name in members(step):
                    if (work/rel/name).is_file():
                        files[(rel/name).as_posix()]=work/rel/name
    for lane,p in extra:
        for name in ("console.log","run.json","health.json","training.jsonl","checkpoint.pt","probe.json"):
            if (p/name).is_file():
                files[f"interrupted/{lane}/{name}"]=p/name
    if (work/"comparison.json").is_file() and tag=="review":
        files["comparison.json"]=work/"comparison.json"
    record=dict(tag=tag,experiment=experiment,source_review_sha256=review_sha,parent_checkpoint=start_sha,
                scope="training-only diagnostic forks",settings=lanes,device_names=[torch.cuda.get_device_name(i) for i in range(2)],
                torch_version=str(torch.__version__),cuda=torch.version.cuda,sha256={k:sha(v) for k,v in files.items()})
    target=outputs/f"chair-s42-stage12-{tag}.zip"
    tmp=target.with_suffix(".tmp")
    with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for n,p in files.items():z.write(p,n)
        z.writestr("stage12-review.json",json.dumps(record,indent=2,allow_nan=False))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None and set(z.namelist())==set(files)|{"stage12-review.json"}
    tmp.replace(target)
    print(f"Saved {target.name} ({target.stat().st_size/1e6:.1f} MB)",flush=True)
    encoded=base64.b64encode(target.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{target.name}" href="data:application/zip;base64,{encoded}">Download {target.name}</a>'))

In [ ]:
def milestone(stop):
    attempts={}
    processes={}
    with ExitStack() as stack:
        try:
            for i,lane in enumerate(lanes):
                dest=work/relative(lane,stop)
                if dest.exists():
                    checked(dest,lane,stop)
                    print(f"Reusing {lane} at {stop}.")
                    continue
                run=Path(stack.enter_context(TemporaryDirectory(dir=work)))
                attempts[lane]=run
                seed_copy(lane,stop,run)
                env=dict(os.environ,CUDA_VISIBLE_DEVICES=devices[i],OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",
                         PYTHONPATH=os.pathsep.join([str(repo),os.environ.get("PYTHONPATH","")]))
                writer=stack.enter_context((run/"console.log").open("ab"))
                reader=stack.enter_context((run/"console.log").open(encoding="utf-8",errors="replace"))
                cmd=[sys.executable,"-u",str(worker_path),str(run),str(manifest),str(stop)]
                proc=subprocess.Popen(cmd,cwd=repo,env=env,stdout=writer,stderr=subprocess.STDOUT)
                processes[lane]=(proc,reader)
            while processes:
                for lane,(proc,reader) in list(processes.items()):
                    code=proc.poll()
                    for line in reader:
                        print(f"[{lane}] {line.rstrip()}",flush=True)
                    if code is None:
                        continue
                    if code:
                        raise RuntimeError(f"{lane} exited with status {code}")
                    run=attempts[lane];checked(run,lane,stop)
                    dest=work/relative(lane,stop)
                    dest.parent.mkdir(exist_ok=True,parents=True)
                    with TemporaryDirectory(dir=dest.parent) as tmp:
                        staged=Path(tmp)/"run";staged.mkdir()
                        for name in members(stop):shutil.copyfile(run/name,staged/name)
                        staged.rename(dest)
                    del processes[lane]
                if processes:time.sleep(2)
            observations=[checked(work/relative(lane,stop),lane,stop) for lane in lanes]
            assert observations[0]["rng_sha256"]==observations[1]["rng_sha256"], "Random streams diverged across lanes."
        except BaseException:
            for proc,_ in processes.values():
                if proc.poll() is None:proc.terminate()
            for proc,_ in processes.values():proc.wait()
            snapshot("diagnostics",attempts.items())
            raise

for stop in stops:
    milestone(stop)
    if stop==128:
        snapshot("step128")

summary={}
for lane in lanes:
    summary[lane]={}
    for step in stops:
        p=checked(work/relative(lane,step),lane,step)
        h=p["probes"]["original"]["health"];f=h["fine"]
        summary[lane][str(step)]=dict(fg_psnr=f["region_psnr"],white_psnr=h["white_region_psnr"],
            fg_opacity=f["region_opacity"],colour_gradient_norm=f["color_grad"],dead_on_fixed_rays=dead_field(h),
            passed_foreground_check=p["passed_foreground_check"])
h=summary["full"]["128"]
summary["archived_full_128"]=failure
summary["full_128_reproduced_white_signature"]=bool(h["dead_on_fixed_rays"] and
    abs(h["fg_psnr"]-failure["fine"]["region_psnr"])<.05 and h["fg_opacity"]<.02)
write_json(work/"comparison.json",summary)
print(json.dumps({k:summary[k] for k in lanes},indent=2))
print("Full replay reproduced the recorded white signature at 128:",summary["full_128_reproduced_white_signature"])
snapshot("review")